# 1. Load the dataset

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [263]:
file_path = 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf'
loader = PyPDFLoader(file_path)
document = loader.load()

In [264]:
print(f'Document has : {len(document)} pages ')
print(f'type of whole document : {type(document)}')
print(f'type of one page: {type(document[0])}')
print(f'meta data of first page: {document[0].metadata}')

Document has : 570 pages 
type of whole document : <class 'list'>
type of one page: <class 'langchain_core.documents.base.Document'>
meta data of first page: {'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-10-31T15:40:29+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-10-31T12:10:13-04:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf', 'total_pages': 570, 'page': 0, 'page_label': 'Cover'}


## 1.2 split the dataset 

In [265]:
chapter_starts = [
    (24, 'Chapter 1. The Machine Learning Landscape'),
    (54, 'Chapter 2. End-to-End Machine Learning Project'),
    (102, 'Chapter 3. Classification'),
    (128, 'Chapter 4. Training Models'),
    (168, 'Chapter 5. Support Vector Machines'),
    (190, 'Chapter 6. Decision Trees'),
    (204, 'Chapter 7. Ensemble Learning and Random Forests'),
    (228, 'Chapter 8. Dimensionality Reduction'),
    (252, 'Chapter 9. Up and Running with TensorFlow'),
    (276, 'Chapter 10. Introduction to Artificial Neural Networks'),
    (298, 'Chapter 11. Training Deep Neural Nets'),
    (338, 'Chapter 12. Distributing TensorFlow Across Devices and Servers'),
    (378, 'Chapter 13. Convolutional Neural Networks'),
    (404, 'Chapter 14. Recurrent Neural Networks'),
    (438, 'Chapter 15. Autoencoders'),
    (498, 'Appendix A. Exercise Solutions'),
    (524, 'Appendix B. Machine Learning Project Checklist'),
    (530, 'Appendix C. SVM Dual Problem'),
    (534, 'Appendix D. Autodiff'),
    (542, 'Appendix E. Other Popular ANN Architectures'),
]

def get_chapter(page_number):
    chapter = 'Front Matter'
    for start_page, title in chapter_starts:
        if page_number >= start_page:
            chapter = title
        else:
            break
    return chapter

filter_doc = []
for page in document:
    chapter = get_chapter(page.metadata['page'])
    # add chapter info to metadata
    page_label = page.metadata.get('page_label', page.metadata['page'] + 1)
    page.metadata['chapter'] = chapter
    page.metadata['metadata_label'] = f"{chapter} | page {page_label}"
    filter_doc.append(page)


In [266]:
len(filter_doc)
filter_doc[120].metadata

{'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)',
 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)',
 'creationdate': '2017-10-31T15:40:29+00:00',
 'author': 'Aurélien Géron',
 'moddate': '2017-10-31T12:10:13-04:00',
 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow',
 'trapped': '/False',
 'source': 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf',
 'total_pages': 570,
 'page': 120,
 'page_label': '99',
 'chapter': 'Chapter 3. Classification',
 'metadata_label': 'Chapter 3. Classification | page 99'}

# 2. Chunking the dataset into smaller pieces

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    add_start_index=True
)
chunks = text_splitter.split_documents(filter_doc)


In [268]:
def clean_text(text):
    return text.encode("utf-8", "ignore").decode("utf-8")

for chunk in chunks:
    chunk.page_content = clean_text(chunk.page_content)

In [269]:
print(f'the length of chunks {len(chunks)}')
print(f'the type of chunks : {type(chunks)}')
print(f'the type of one chunk: {type(chunks[0])}')
print(f'there are total {sum([len(chunk.page_content.lower()) for chunk in chunks])} characters in all chunks')

the length of chunks 1470
the type of chunks : <class 'list'>
the type of one chunk: <class 'langchain_core.documents.base.Document'>
there are total 1219208 characters in all chunks


# 3. Generate embeddings for each chunk

In [270]:
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

In [271]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small'
)

# 4. store the embeddings in a vector database

In [272]:
# Create an FAISS vectore store
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
    docstore=InMemoryDocstore(),
)


## 4.1 test the vector database by querying with a sample question

In [273]:
question = 'what is random forest'
retrieved_docs  = vector_store.similarity_search(query=question,k=10)
retriever_simple = vector_store.as_retriever(
    search_type = 'mmr',
    search_kwargs={"k": 5, "fetch_k": 20}
    )


# 5. Initialize the LLM and set up the retrieval-based question-answering system

## 5.1 Design and implement Multi-stage Retrieval and Re-ranking (MMR) strategy

In [274]:
# Multi-stage Retrieval + rerank
def retrieve_candidates(query,k):

    return vector_store.similarity_search(query,k=20)


rerank_prompt = ChatPromptTemplate.from_template(
    """
    You are a ranking assistant.

    Given a question and a list of documents, rank the documents by relevance.
    Return ONLY a valid JSON list of indices.
    Do NOT use markdown.

    Example:
    [0, 3, 5, 2, 1]

    Question:
    {question}

    Documents:
    {documents}

    Return the indices of the top {top_k} most relevant documents in order.
    """)

rerank_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [275]:
import json
def parse_indices(text):
    import json, re

    # 1. remove markdown
    text = re.sub(r"```json|```", "", text).strip()

    # 2. load json
    try:
        return json.loads(text)
    except:
        # 3. fallback
        numbers = re.findall(r"\d+", text)
        return [int(n) for n in numbers]

In [276]:
def rerank_docs(documents,query,top_k):
    format_text  = "\n\n" .join(f"[{i}]\n{doc.metadata.get('metadata_label', 'Unknown source')}\n{doc.page_content}" for i, doc in enumerate(documents))
    response  = rerank_llm.invoke(
        rerank_prompt.format(
            question = query,
            documents = format_text,
            top_k = top_k
        )
    )
    indices = parse_indices(response.content)
    return [documents[i] for i in indices]


In [277]:
result_docs = rerank_docs(retrieved_docs,question,5)

In [278]:
def multi_stage_retrieval(query):
    # Stage 1: recall
    candidates = retrieve_candidates(query, k=20)

    # Stage 2: rerank
    top_docs = rerank_docs(candidates,query,top_k=10)

    return top_docs

In [279]:
# prompt design for LLM to answer
# citation and source
# hallucination
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a machine learning expert and tutor.

        Your task is to answer the question using ONLY the provided context.

        Guidelines:
        - Use ONLY the information from the context
        - Do NOT use your own knowledge
        - If the answer is not in the context, say: "I don't know based on the provided document"
        - Explain concepts simply
        - Provide examples only if they are supported by the context
        - ALWAYS cite your sources using this format:
          (Source: Chapter X | page Y)
        - If multiple sources are used, cite all of them
        - Be clear and concise

        """
    ),
    (
        "human",
        """
        Context:
        {context}

        Question:
        {question}

        Answer:
        """
    ),
])

In [280]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(model="gpt-4o-mini")

retriever = RunnableLambda(multi_stage_retrieval)

def format_docs(docs):
    return "\n\n".join(
        f"Source:[{doc.metadata.get('metadata_label', 'Unknown source')}]\n Content:{doc.page_content}" for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [281]:
response = rag_chain.invoke("how to solve poor quality data?")
display(Markdown(response))


To solve poor quality data, it's important to take several steps:

1. **Clean the Data**: Spend time cleaning your training data, as many data scientists do. This includes:
   - Discarding or fixing clearly outlier instances manually.
   - Handling missing values by deciding whether to ignore the attribute, ignore those instances, fill in missing values (e.g., with the median), or train different models with and without the feature.

2. **Focus on Relevant Features**: Ensuring that the training data includes relevant features and not too many irrelevant ones is crucial. This involves:
   - **Feature Selection**: Choosing the most useful features among existing ones.
   - **Feature Extraction**: Combining existing features to create more useful ones.
   - **Creating New Features**: Gathering additional data that could be beneficial.

3. **Reduce Noise**: Identify and reduce noise in the data, such as outliers and measurement errors.

By implementing these strategies, the quality of the data can be improved, which in turn enhances the performance of the system (Source: Chapter 1 | pages 25-26).

# 6. Evalaution of the system

## 6.1 Build a small benchmark set
Use a fixed set of questions so you can compare retrieval and answer quality after every change.

In [30]:
eval_set = [
    {
        "question": "What is unsupervised learning?",
        "expected_chapter": "Chapter 1",
        "expected_terms": ["unlabeled", "patterns", "without a teacher"],
    },
    {
        "question": "What are the main steps in a machine learning project?",
        "expected_chapter": "Chapter 2",
        "expected_terms": ["frame the problem", "get the data", "select and train a model"],
    },
    {
        "question": "What is the ROC curve used for?",
        "expected_chapter": "Chapter 3",
        "expected_terms": ["classifier", "tradeoff", "true positive"],
    },
    {
        "question": "What is the difference between batch gradient descent and stochastic gradient descent?",
        "expected_chapter": "Chapter 4",
        "expected_terms": ["entire training set", "single instance", "gradient descent"],
    },
    {
        "question": "What is a support vector machine?",
        "expected_chapter": "Chapter 5",
        "expected_terms": ["svm", "classification", "regression"],
    },
    {
        "question": "How do decision trees make predictions?",
        "expected_chapter": "Chapter 6",
        "expected_terms": ["root node", "leaf", "split"],
    },
    {
        "question": "What is a random forest?",
        "expected_chapter": "Chapter 7",
        "expected_terms": ["decision trees", "ensemble", "random forest"],
    },
    {
        "question": "Why do we use dimensionality reduction?",
        "expected_chapter": "Chapter 8",
        "expected_terms": ["curse of dimensionality", "speed", "memory"],
    },
    {
        "question": "What is TensorFlow?",
        "expected_chapter": "Chapter 9",
        "expected_terms": ["open source", "numerical computation", "machine learning"],
    },
    {
        "question": "What is an autoencoder?",
        "expected_chapter": "Chapter 15",
        "expected_terms": ["neural network", "representation", "coding"],
    },
]

len(eval_set)


10

## 6.2 Run retrieval and answer evaluation
For each question, measure whether retrieval found the expected chapter and whether the final answer contains the expected chapter citation and key terms.

In [31]:
from pprint import pprint

def extract_source_labels(docs):
    return [doc.metadata.get("metadata_label", "Unknown source") for doc in docs]

def evaluate_one_question(item):
    question = item["question"]
    expected_chapter = item["expected_chapter"]
    expected_terms = item.get("expected_terms", [])
    docs = multi_stage_retrieval(question)
    retrieved_sources = extract_source_labels(docs)
    answer = rag_chain.invoke(question)

    retrieval_hit = any(expected_chapter in source for source in retrieved_sources)
    citation_hit = expected_chapter in answer
    matched_terms = [term for term in expected_terms if term.lower() in answer.lower()]

    return {
        "question": question,
        "expected_chapter": expected_chapter,
        "retrieved_sources": retrieved_sources,
        "retrieval_hit": retrieval_hit,
        "answer": answer,
        "citation_hit": citation_hit,
        "matched_terms": matched_terms,
        "term_hit_rate": round(len(matched_terms) / max(len(expected_terms), 1), 2),
    }

eval_results = [evaluate_one_question(item) for item in eval_set]
len(eval_results)


10

In [32]:
total_questions = len(eval_results)
retrieval_hit_rate = sum(item["retrieval_hit"] for item in eval_results) / total_questions
citation_hit_rate = sum(item["citation_hit"] for item in eval_results) / total_questions
avg_term_hit_rate = sum(item["term_hit_rate"] for item in eval_results) / total_questions

summary = {
    "total_questions": total_questions,
    "retrieval_hit_rate@top_docs": round(retrieval_hit_rate, 2),
    "citation_hit_rate": round(citation_hit_rate, 2),
    "average_term_hit_rate": round(avg_term_hit_rate, 2),
}

summary


{'total_questions': 10,
 'retrieval_hit_rate@top_docs': 1.0,
 'citation_hit_rate': 0.9,
 'average_term_hit_rate': 0.83}

In [33]:
for item in eval_results:
    pprint({
        "question": item["question"],
        "expected_chapter": item["expected_chapter"],
        "retrieval_hit": item["retrieval_hit"],
        "citation_hit": item["citation_hit"],
        "matched_terms": item["matched_terms"],
        "retrieved_sources": item["retrieved_sources"][:3],
    })
    print("-" * 80)


{'citation_hit': True,
 'expected_chapter': 'Chapter 1',
 'matched_terms': ['unlabeled', 'patterns', 'without a teacher'],
 'question': 'What is unsupervised learning?',
 'retrieval_hit': True,
 'retrieved_sources': ['Chapter 1. The Machine Learning Landscape | page 10',
                       'Chapter 1. The Machine Learning Landscape | page 12',
                       'Chapter 1. The Machine Learning Landscape | page 11']}
--------------------------------------------------------------------------------
{'citation_hit': False,
 'expected_chapter': 'Chapter 2',
 'matched_terms': ['frame the problem', 'get the data'],
 'question': 'What are the main steps in a machine learning project?',
 'retrieval_hit': True,
 'retrieved_sources': ['Appendix B. Machine Learning Project Checklist | page '
                       '503',
                       'Chapter 2. End-to-End Machine Learning Project | page '
                       '33',
                       'Chapter 2. End-to-End Machine Learnin

## 6.3 How to use these results
- If `retrieval_hit_rate` is low, improve chunking, metadata, or retrieval settings first.
- If retrieval is good but `citation_hit_rate` or `term_hit_rate` is low, improve the answer prompt or reranking.
- Keep this benchmark fixed and rerun it after every major RAG change.